# 🛡️ TransferGuard: Detecting and Avoiding Negative Transfer in Lightweight Vision Models

> **Computer Vision Course Project**  
> **Group Members:**  
> - **Ghanishth Saini** (Roll: 24121)  
> - **Arvind Yadav** (Roll: 24059)  
> - **Aakash Jagtap** (Roll: 24157)  

---

### 📌 Overview & Resumable Cloud Execution
This notebook executes the complete experimental benchmark on Google Colab GPU (T4/V100/A100).

✨ **Zero Progress Lost:** All training commands feature **automatic checkpoint resumption** (`--resume auto`) and **Google Drive persistence** (`--drive-backup auto`). If your Colab runtime disconnects or restarts midway through training, re-running the cells will automatically detect `latest_model.pth` and seamlessly continue from the exact epoch where it was interrupted!

**Hardware Acceleration Setup:**
Go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU** before executing the cells below.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

## 2. (Recommended) Mount Google Drive for Automatic Zero-Loss Checkpoint Persistence
Mounting Google Drive enables **automatic bidirectional synchronization**:
- Every epoch, `latest_model.pth`, `best_model.pth`, and `train_history.json` are automatically copied to your Drive at `/content/drive/MyDrive/TransferGuard_Backups/`.
- If Google Colab disconnects or wipes the ephemeral workspace, re-running the cell will **automatically restore** previous weights from Drive and seamlessly continue training!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/TransferGuard_Backups
print("✅ Google Drive successfully mounted! Checkpoint auto-sync is ready.")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone repository from GitHub
!git clone https://github.com/arvindyadav186/TransferGuard-Detecting-and-Avoiding-Negative-Transfer-in-Lightweight-Vision-Models.git
%cd TransferGuard-Detecting-and-Avoiding-Negative-Transfer-in-Lightweight-Vision-Models
!pip install -q -r requirements.txt
!pip install -q -e .
print("Repository cloned and installed successfully!")

## 4. Run Automated Test Suite (Sanity Check)
Verifies all 12 unit tests (Linear CKA invariance, layer activation hooks, dynamic freezing, resumable training, and Google Drive auto-backup/restore).

In [ ]:
!pytest tests/ -v

## 5. Layer-wise CKA Representation Alignment Analysis
Computes Centered Kernel Alignment (CKA) between ImageNet pre-trained representations and target domain representations to detect the "divergence cliff".

In [ ]:
# 1. SVHN (MobileNetV3-Small)
!python scripts/run_cka_analysis.py --dataset svhn --model mobilenet_v3_small --device auto --subset 500

# 2. SVHN (ShuffleNetV2 - Architecture Generalization)
!python scripts/run_cka_analysis.py --dataset svhn --model shufflenet_v2_x1_0 --device auto --subset 500

# 3. CIFAR-100 (MobileNetV3-Small)
!python scripts/run_cka_analysis.py --dataset cifar100 --model mobilenet_v3_small --device auto --subset 500

# 4. Oxford Flowers-102 (MobileNetV3-Small)
!python scripts/run_cka_analysis.py --dataset flowers102 --model mobilenet_v3_small --device auto --subset 500

from IPython.display import Image, display
display(Image("results/cka/cka_mobilenet_v3_small_svhn.png"))
display(Image("results/cka/cka_shufflenet_v2_x1_0_svhn.png"))


## 6. Train Baselines & Proposed TransferGuard Across Benchmarks (With Auto-Resume & Drive Sync)
All training commands include `--resume auto --drive-backup auto`:
- **Mixed Precision (AMP):** Automatically utilizes Tesla T4 Tensor Cores for maximum throughput.
- **Drive Sync:** Every epoch is mirrored to Google Drive. If disconnected, simply re-run the cell to resume without any wasted compute.
- **Configs Available:**
  - `configs/svhn_mobilenetv3.yaml` (SVHN - MobileNetV3 - Severe Mismatch)
  - `configs/svhn_shufflenetv2.yaml` (SVHN - ShuffleNetV2 - Architecture Generalization)
  - `configs/cifar100_mobilenetv3.yaml` (CIFAR-100 - MobileNetV3 - Moderate Shift)
  - `configs/flowers102_mobilenetv3.yaml` (Oxford Flowers-102 - MobileNetV3 - High Scarcity)


In [ ]:
# 1. Baseline: Train from Scratch (Random Initialization)
# Change --config to any of the 4 YAMLs above
!python scripts/train_scratch.py --config configs/svhn_mobilenetv3.yaml --epochs 30 --device auto --resume auto --drive-backup auto


In [ ]:
# 2. Baseline: Standard Fine-Tuning
!python scripts/train_finetune.py --config configs/svhn_mobilenetv3.yaml --mode full --epochs 30 --device auto --resume auto --drive-backup auto


In [ ]:
# 3. Baseline: Linear Probing (Frozen Backbone - Negative Transfer on SVHN)
!python scripts/train_finetune.py --config configs/svhn_mobilenetv3.yaml --mode linear_probe --epochs 30 --device auto --resume auto --drive-backup auto


In [ ]:
# 4. Proposed Method: TransferGuard Adaptive Routing & Dynamic Freezing
!python scripts/train_transferguard.py --config configs/svhn_mobilenetv3.yaml --threshold 0.70 --epochs 30 --device auto --resume auto --drive-backup auto


## 7. Comprehensive Benchmark Evaluation & Transfer Gain Table
Computes Transfer Gain $TG = Acc_{transfer} - Acc_{scratch}$ and checks for negative transfer across all datasets.

In [ ]:
!python scripts/evaluate_all.py

## 8. Generate 300 DPI Publication & Presentation Figures
Renders high-resolution publication figures dynamically prioritized using live GPU training metrics and CKA calculations.

In [ ]:
!python scripts/plot_results.py --output-dir results/figures

from IPython.display import Image, display
print("--- Figure 1: Transfer Gain Comparison Across All Benchmarks ---")
display(Image("results/figures/transfer_gain_comparison.png"))

print("--- Figure 2: Layer-wise CKA Representational Divergence Curves ---")
display(Image("results/figures/cka_divergence_curve.png"))

print("--- Figure 3: TransferGuard Routing Profiles (SVHN MobileNetV3 vs ShuffleNetV2) ---")
display(Image("results/figures/transferguard_svhn_mobilenetv3_routing_profile.png"))
display(Image("results/figures/transferguard_svhn_shufflenetv2_routing_profile.png"))

print("--- Figure 4: Training Convergence Trajectories ---")
display(Image("results/figures/training_convergence_svhn_mobilenetv3.png"))
display(Image("results/figures/training_convergence_svhn_shufflenetv2.png"))


## 9. Run Interactive Live Terminal Demo

In [ ]:
!python scripts/demo_live.py

## 10. Package Figures & Checkpoints for Presentation

In [ ]:
!zip -r transferguard_presentation_results.zip results/figures/ results/benchmark_summary.csv docs/
from google.colab import files
files.download("transferguard_presentation_results.zip")